# 02 · Curry-Howard: demostrar fórmulas lógicas escribiendo programas

> **En el examen:** es siempre el **Ejercicio 1** (2 puntos). Sale en **todos** los
> exámenes desde 20-21. Tiene dos apartados:
> - **a)** una fórmula de lógica **intuicionista** (1 punto),
> - **b)** una fórmula de lógica **clásica**, usando como ayuda el **tercio excluso** o la
>   **doble negación** (1 punto).
>
> Es el ejercicio más "mecánico" del examen: con el método de este archivo se saca casi siempre.

Antes de seguir, ten claro del archivo 01: lambdas (`x => ...`), tuplas, `Either`
(`Left`/`Right`), `match` y el tipo `Nothing`.

---

### Preámbulo
Ejecuta la siguiente celda antes que las demás. Las celdas de código se pueden ejecutar en orden; los fragmentos que no son ejecutables por sí solos se muestran como texto.

In [ ]:
// Preámbulo: ejecuta esta celda primero (las mismas definiciones que vienen en los exámenes)
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:
    def from(a: A): B
    def to(b: B): A
    def equalA(a1: A, a2: A): Boolean = a1 == a2
    def equalB(b1: B, b2: B): Boolean = b1 == b2
    def law1(a: A): Boolean = equalA(to(from(a)), a)
    def law2(b: B): Boolean = equalB(from(to(b)), b)

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:
    def void[A]: Tree[A] = Empty()
    def leaf[A](a: A): Node[A] = Node(Empty(), a, Empty())
    def right[A](a: A, tree: Tree[A]): Node[A] = Node(Empty(), a, tree)
    def left[A](tree: Tree[A], a: A): Node[A] = Node(tree, a, Empty())
    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] = Node(left, a, right)
    def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
        tree match
            case Empty() => empty
            case Node(left, a, right) => node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

import Tree._

def pow(base: Int, exp: Int): Int = math.pow(base, exp).toInt

: 

## 1. La idea en una frase

> **Demostrar una fórmula = escribir un programa cuyo tipo sea esa fórmula.**

Si consigues escribir una función de Scala que compile con ese tipo (sin trampas como `???`
o recursión infinita), la fórmula es un teorema. El compilador hace de corrector.

---

## 2. El diccionario: lógica ↔ tipos

| Lógica | Se lee | Scala | Cómo se **construye** (introducción) | Cómo se **usa** (eliminación) |
|---|---|---|---|---|
| p → q | "si p entonces q" | `P => Q` | una lambda: `(p: P) => ...` | aplicarla: `f(p)` |
| p ∧ q | "p y q" | `(P, Q)` | un par: `(p, q)` | `t._1`, `t._2` o `case (p, q)` |
| p ∨ q | "p o q" | `Either[P, Q]` | `Left(p)` o `Right(q)` | `match` con dos casos |
| ⊥ | "falso / absurdo" | `Nothing` | no se puede construir de la nada | un valor de tipo `Nothing` vale como **cualquier** tipo |
| ⊤ | "verdadero" | `Unit` | `()` | — |
| ¬p | "no p" | `Not[P]` = `P => Nothing` | una lambda `p => (algo de tipo Nothing)` | aplicarla a una `p`: `np(p)` da `Nothing` |
| p ↔ q | "p si y solo si q" | `P <=> Q` = `(P => Q, Q => P)` | un par de funciones | `._1`, `._2` |

Los alias vienen en el preámbulo del examen:

In [ ]:
type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

**¿Por qué ¬p es `P => Nothing`?** Porque "no p" significa "si me das una prueba de p,
llego a una contradicción (⊥)". Y como `Nothing` no tiene valores, una función
`P => Nothing` solo puede existir si nunca te pueden dar una `P` de verdad.

---

## 3. Paso 1: traducir la fórmula a un tipo

### Precedencia de los operadores (¡importantísimo!)

De **más fuerte** (se agrupa primero) a **más débil**:

1. `¬`
2. `∧`
3. `∨`
4. `→` (y además **asocia a la derecha**: `a → b → c` es `a → (b → c)`)

Antes de traducir, **pon tú los paréntesis**. Ejemplos sacados de exámenes:

| Fórmula | Con paréntesis | Tipo Scala |
|---|---|---|
| ¬p ∨ ¬q → ¬(p ∧ q) | (¬p ∨ ¬q) → ¬(p ∧ q) | `Either[Not[P], Not[Q]] => Not[(P, Q)]` |
| (a → b ∨ c) → (b → ¬a) ∧ ¬c → ¬a | (a → (b ∨ c)) → ( ((b → ¬a) ∧ ¬c) → ¬a ) | `(A => Either[B, C]) => ((B => Not[A], Not[C])) => Not[A]` |
| (p → q) → ¬q → ¬p | (p → q) → (¬q → ¬p) | `(P => Q) => Not[Q] => Not[P]` |
| p ∧ q → r | (p ∧ q) → r | `((P, Q)) => R` |

Fíjate en el último: "p ∧ q → r" es una función que recibe **un par**, así que el tipo es
`((P, Q)) => R` con **doble paréntesis**. `(P, Q) => R` sería una función de dos
argumentos (archivo 01, sección 4).

### Argumentos con premisas: `{premisas} ⊢ conclusión`

Si te dan un **argumento** en vez de una fórmula, las premisas se convierten en
**parámetros** de la función, y la conclusión en el tipo de retorno:

$\{ p \rightarrow q,\; r \vee s,\; s \rightarrow \neg q,\; \neg r \} \vdash \neg p$

In [ ]:
def proof[P, Q, R, S](p1: P => Q,
                      p2: Either[R, S],
                      p3: S => Not[Q],
                      p4: Not[R]): Not[P] = ???

Las letras de las proposiciones (p, q, r…) se convierten en **parámetros de tipo** en
mayúscula (`[P, Q, R, S]`).

---

## 4. Paso 2: construir el programa (el juego de "meta" y "recursos")

En cada momento tienes:

- una **meta**: el tipo que tienes que fabricar (lo que hay que demostrar),
- unos **recursos**: las variables que tienes a mano (lo que puedes usar).

Escribe `???` donde aún te falta algo y ve rellenando. En cada paso haz **una** de estas
jugadas:

### Jugadas según la META

| Si la meta es… | Haz… |
|---|---|
| `P => Q` | escribe una lambda `p => ???`. Ganas el recurso `p: P`; la nueva meta es `Q` |
| `Not[P]` (= `P => Nothing`) | igual: `p => ???`. Ganas `p: P`; la meta pasa a ser `Nothing` |
| `(P, Q)` | escribe `(???, ???)` y resuelve cada mitad por separado |
| `Either[P, Q]` | `Left(???)` (meta `P`) o `Right(???)` (meta `Q`). **Espera** a elegir lado hasta que sepas cuál puedes conseguir (normalmente tras hacer un `match` sobre un recurso) |
| `Nothing` | busca una negación entre los recursos, `np: Not[X]`, y aplícala: `np(???)` con meta `X` |
| cualquier cosa, y tienes algo de tipo `Nothing` | ¡ya está! Un `Nothing` encaja en cualquier tipo |

### Jugadas según los RECURSOS

| Si tienes… | Puedes… |
|---|---|
| `f: P => Q` y un `p: P` | obtener `f(p): Q` |
| `f: P => Q => R` | `f(p)(q): R` (aplícala dos veces) |
| `t: (P, Q)` | sacar `t._1: P` y `t._2: Q` |
| `e: Either[P, Q]` | hacer `e match { case Left(p) => ???; case Right(q) => ??? }`. **Las dos ramas tienen la misma meta** |
| `np: Not[P]` y un `p: P` | obtener `np(p): Nothing` (y con eso, cualquier cosa) |

### Consejo de orden

1. Primero **introduce** todas las flechas de la meta con lambdas (`f => g => h => ...`),
   para tener todos los recursos.
2. Si algún recurso es un `Either`, suele ser buena idea hacerle `match` pronto.
3. Luego trabaja "hacia atrás" desde la meta: ¿qué función de mis recursos devuelve el tipo
   que necesito?

---

## 5. Ejemplos resueltos (lógica intuicionista, apartado a)

### Ejemplo 1 (calentamiento): ⊢ (p → q) → (q → r) → p → r

Tipo: `(P => Q) => (Q => R) => P => R`

In [ ]:
def proof[P, Q, R]: (P => Q) => (Q => R) => P => R =
    f => g => p =>     // recursos: f: P=>Q, g: Q=>R, p: P.   meta: R
        g(f(p))        // f(p): Q, y g de eso: R

### Ejemplo 2 (examen 25-26 Vicálvaro): ⊢ ¬p ∨ ¬q → ¬(p ∧ q)

Tipo: `Either[Not[P], Not[Q]] => Not[(P, Q)]`

Paso a paso:

In [ ]:
// Paso 1: la meta es una flecha → lambda. Recurso e: Either[Not[P], Not[Q]]
// Meta: Not[(P, Q)]
def proof[P, Q]: Either[Not[P], Not[Q]] => Not[(P, Q)] =
    e => ???

// Paso 2: e es un Either → match. Cada rama tiene meta Not[(P, Q)]
    e => e match
        case Left(np)  => ???    // np: Not[P]
        case Right(nq) => ???    // nq: Not[Q]

// Paso 3: meta Not[(P,Q)] = (P,Q) => Nothing → lambda que recibe el par
        case Left(np)  => (p, q) => np(p)    // np(p): Nothing ✓
        case Right(nq) => (p, q) => nq(q)    // nq(q): Nothing ✓

Solución del examen (más compacta: una función hecha solo de `case`):

In [ ]:
def proof[P, Q]: Either[Not[P], Not[Q]] => Not[(P, Q)] =
    case Left(np) => (p, _) => np(p)
    case Right(nq) => (_, q) => nq(q)

`(p, _) => ...` recibe el par y como no usamos el segundo componente lo llamamos `_`.

### Ejemplo 3 (examen 25-26 Móstoles): ⊢ (a → b ∨ c) → (b → ¬a) ∧ ¬c → ¬a

Tipo (con los paréntesis bien puestos, sección 3):
`(A => Either[B, C]) => ((B => Not[A], Not[C])) => Not[A]`

In [ ]:
def proof[A, B, C]: (A => Either[B, C]) => ((B => Not[A], Not[C])) => Not[A] =
    f => (g, nc) => a =>      // f: A=>Either[B,C]; g: B=>Not[A]; nc: Not[C]; a: A
                              // meta: Nothing (porque Not[A] = A => Nothing y ya metimos la a)
        f(a) match            // f(a): Either[B, C] → match
            case Left(b)  => g(b)(a)   // g(b): Not[A], aplicado a a: Nothing ✓
            case Right(c) => nc(c)     // nc(c): Nothing ✓

Observa `(g, nc) => ...`: el segundo argumento es un **par**, y lo desempaquetamos
directamente en dos nombres (Scala 3 lo permite).

### Ejemplo 4 (con premisas, examen 22-23 Móstoles):

### { p → q ∨ r, ¬r, q → s ∨ t, ¬(s ∨ t) } ⊢ ¬p

In [ ]:
def proof[P, Q, R, S, T](p1: P => Either[Q, R],
                         p2: Not[R],
                         p3: Q => Either[S, T],
                         p4: Not[Either[S, T]]): Not[P] =
    (p: P) =>                   // meta: Nothing
        p1(p) match
            case Left(q)  => p4(p3(q))   // p3(q): Either[S,T]; p4 de eso: Nothing
            case Right(r) => p2(r)       // Nothing

### Ejemplo 5 (doble negación en la meta, examen 25-26 Vicálvaro Bis):

### ⊢ ¬(p ∧ q) → ¬¬(¬p ∨ ¬q)

Tipo: `Not[(P, Q)] => Not[Not[Either[Not[P], Not[Q]]]]`

Este es de los difíciles. Recuerda que `Not[Not[X]]` = `(X => Nothing) => Nothing`.

In [ ]:
def proof[P, Q]: Not[(P, Q)] => Not[Not[Either[Not[P], Not[Q]]]] =
    npq => ne =>          // npq: Not[(P,Q)];  ne: Not[Either[Not[P], Not[Q]]]
                          // meta: Nothing
        ne(Left(p =>      // para usar ne necesito un Either. Elijo Left: necesito un Not[P]
                          //   → lambda p => ..., meta Nothing, y ahora tengo p
            ne(Right(q => // vuelvo a usar ne, ahora con Right: necesito un Not[Q] → q => ...
                npq((p, q))   // ¡tengo p y q! npq((p, q)): Nothing ✓
            ))
        ))

**Truco clave:** cuando tienes una negación `ne` de un `Either` y la meta es `Nothing`,
puedes **usar `ne` varias veces**. Eliges un lado, y si en ese lado te falta algo, vuelves a
llamar a `ne` con el otro lado. Así acabas teniendo los dos datos a la vez.

### Ejemplo 6 (equivalencia y ⊥, examen 23-24 Extraordinaria): ⊢ p ↔ p ∨ ⊥

Tipo: `P <=> Either[P, Nothing]`, es decir, un par de funciones.

In [ ]:
def proof[P]: P <=> Either[P, Nothing] =
    (p => Left(p),             // ida: P => Either[P, Nothing]
     {                         // vuelta: Either[P, Nothing] => P
        case Left(p)  => p
        case Right(n) => n     // n: Nothing → vale como P ✓
     })

---

## 6. Lógica clásica (apartado b): tercio excluso y doble negación

Algunas fórmulas son ciertas en lógica clásica pero **no** se pueden demostrar sin ayuda
(por ejemplo `p ∨ ¬p`). El enunciado te deja usar una de estas dos leyes como premisa extra,
**aplicada a una proposición concreta que te indican**.

### 6.1 Tercio excluso (LEM): p ∨ ¬p

En el preámbulo:

In [ ]:
trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

Si tienes `lem: ExcludedMiddle`, escribir **`lem[X]`** te da un `Either[X, Not[X]]`:
"o X es verdad, o no lo es". Lo usas haciendo `match`:

```scala
lem[X] match
    case Left(x)   => ???   // caso en que tengo una x
    case Right(nx) => ???   // caso en que tengo una Not[X]
```

En exámenes antiguos el tercio excluso te lo dan ya como parámetro:
`lemp: Either[P, Not[P]]`, y haces `lemp match ...` directamente (sin `[P]`).

**Receta con LEM:** haz `match` sobre el tercio excluso de la proposición que te digan, al
principio. En la rama `Left` normalmente es fácil; en la rama `Right` usa la negación.

#### Ejemplo 7 (25-26 Móstoles): ⊢ (a → b) → ¬a ∨ b, con LEM en a

In [ ]:
def proof[A, B](lem: ExcludedMiddle): (A => B) => Either[Not[A], B] =
    f => lem[A] match
        case Left(a)   => Right(f(a))   // tengo a: puedo dar b
        case Right(na) => Left(na)      // tengo ¬a: lo devuelvo tal cual

#### Ejemplo 8 (25-26 Vicálvaro): la misma fórmula, pero con LEM en **b**

In [ ]:
def proof[A, B](lem: ExcludedMiddle): (A => B) => Either[Not[A], B] =
    f => lem[B] match
        case Left(b)   => Right(b)
        case Right(nb) => Left(a => nb(f(a)))   // construyo ¬a: si me das a, f(a): B, y nb lo contradice

Compara los ejemplos 7 y 8: la **misma fórmula** se resuelve distinto según sobre qué
proposición te dejen usar el tercio excluso. **Usa exactamente la que diga el enunciado.**

#### Ejemplo 9 (25-26 Extraordinaria): ⊢ (p → q) ∨ (q → p), con LEM en q

In [ ]:
def proof[P, Q](lem: ExcludedMiddle): Either[P => Q, Q => P] =
    lem[Q] match
        case Left(q)   => Left(_ => q)   // p → q: ignoro la p y devuelvo la q que ya tengo
        case Right(nq) => Right(nq)      // nq: Q => Nothing sirve como Q => P

¿Por qué `nq` vale como `Q => P`? Porque devuelve `Nothing`, y `Nothing` encaja en cualquier
tipo (en Scala, `Nothing` es "subtipo de todo").

### 6.2 Doble negación (DN): ¬¬p → p

En el preámbulo:

In [ ]:
trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

Si tienes `dn: DoubleNegation`, entonces `dn[X]` es una función `Not[Not[X]] => X`. Para
usarla le pasas una lambda:

```scala
dn[X](nx => ???)     // nx: Not[X].   meta dentro: Nothing
```

**Receta con DN:** si la meta es `X` (la proposición que te indican), escribe
`dn[X](nx => ???)`. Ahora tienes un recurso extra `nx: Not[X]` y la meta es `Nothing`.
Casi siempre se termina con `nx(algo de tipo X)`.

En exámenes antiguos te la dan como parámetro: `dn: Not[Not[P]] => P`, y se usa `dn(np => ...)`.

#### Ejemplo 10 (24-25 Móstoles, ley de Peirce): ⊢ ((p → q) → p) → p, con DN en p

In [ ]:
def proof[P, Q](dn: DoubleNegation): ((P => Q) => P) => P =
    f => dn[P](np =>     // np: Not[P]; meta: Nothing
        np(f(np))        // f necesita un P => Q: np (P => Nothing) sirve. f(np): P. np(eso): Nothing ✓
    )

#### Ejemplo 11 (24-25 Vicálvaro): ⊢ (¬q → ¬p) → (¬q → p) → q, con DN en q

In [ ]:
def proof[P, Q](dn: DoubleNegation): (Not[Q] => Not[P]) => (Not[Q] => P) => Q =
    f => g => dn[Q](nq =>     // nq: Not[Q]; meta: Nothing
        f(nq)(g(nq))          // f(nq): Not[P]; g(nq): P; Not[P] aplicado a P: Nothing ✓
    )

#### Ejemplo 12 (con dos LEM, 22-23 Extraordinaria): ⊢ ¬(¬p ∨ ¬q) → p ∧ q

In [ ]:
def proof[P, Q](lemp: Either[P, Not[P]], lemq: Either[Q, Not[Q]]): Not[Either[Not[P], Not[Q]]] => (P, Q) =
    n => (lemp, lemq) match          // match sobre los dos a la vez
        case (Left(p), Left(q)) => (p, q)
        case (_, Right(nq))     => n(Right(nq))   // Nothing → vale como (P, Q)
        case (Right(np), _)     => n(Left(np))

#### Ejemplo 13 (25-26 Vicálvaro Bis): ⊢ ¬a ∨ b → (a → b), con LEM en b "si hace falta"

Fíjate en el "en caso de que sea necesario": esta fórmula **es** intuicionista, no hace falta
el tercio excluso. (La solución del archivo del examen demuestra por error la fórmula
recíproca, `(a → b) → ¬a ∨ b`; esta es la del enunciado.)

In [ ]:
def proof[A, B]: Either[Not[A], B] => A => B =
    case Left(na) => a => na(a)    // na(a): Nothing → vale como B
    case Right(b) => _ => b

Moraleja: antes de usar el tercio excluso, intenta hacerlo sin él. Si sale, mejor.

---

## 7. Errores típicos

- **Traducir mal la precedencia.** Escribe los paréntesis en papel antes de nada.
- **Confundir `((P, Q)) => R` con `(P, Q) => R`.** "p ∧ q → r" lleva doble paréntesis.
- **Elegir `Left`/`Right` demasiado pronto.** Si la meta es un `Either`, primero haz `match`
  sobre tus recursos `Either`; muchas veces cada rama te dice qué lado elegir.
- **Usar el tercio excluso sobre la proposición equivocada.** El enunciado te dice cuál.
- **Dejar un `???`.** Una prueba con `???` no vale: tiene que compilar sin huecos.
- **Olvidar que `Nothing` vale como cualquier cosa.** Si has llegado a un `Nothing` (por
  ejemplo `np(p)`), esa rama está resuelta, sea cual sea la meta.

---

## 8. Ejercicios para practicar

### De exámenes reales (las soluciones están en `exams/`, abre el archivo **sin** `-Enunciado`)

Intuicionistas (apartado a):

| Fórmula | Examen |
|---|---|
| ⊢ (p ∨ q → r) → (r → s) → (p → s) ∧ (q → s) | 25-26 Extraordinaria |
| ⊢ ((q → r) → p → q) → (r → s) → ((p → q) → r) → s | 24-25 Extraordinaria |
| ⊢ (q → r → s) → (p → q) → (p → r) → p → s | 24-25 Móstoles |
| ⊢ (p → q → r) → (s → t → p) → s → t → (u → v → q) → u → v → r | 24-25 Vicálvaro |
| ⊢ ¬¬p ∨ ¬¬q → ¬¬(p ∨ q) | 23-24 Vicálvaro |
| { r → s, p ∨ q, ¬(¬p → s) } ⊢ q ∧ ¬r | 23-24 Móstoles |
| ⊢ ¬¬(p ∨ ¬p) | 22-23 Vicálvaro Especial |
| ⊢ p ∧ q → ¬(p → ¬q) | 22-23 Extraordinaria |
| { p ∨ q ∨ r, p → s, ¬r → ¬s } ⊢ ¬r → q | 22-23 Vicálvaro |
| ⊢ ¬p → ¬¬¬p | 21-22 Vicálvaro |
| ⊢ ¬(p ∨ q) → ¬p ∧ ¬q  y su recíproca | 20-21 Extraordinaria |
| Dilemas constructivos y destructivos | 20-21 Ordinaria |

Clásicas (apartado b):

| Fórmula | Ayuda | Examen |
|---|---|---|
| ⊢ ¬a ∨ b → (a → b) | LEM en b "si hace falta" | 25-26 Vicálvaro Bis |
| ⊢ (p → q ∨ r) → (p → q) ∨ (p → r) | LEM en r | 24-25 Extraordinaria |
| ⊢ ¬¬p → p ... ¬s → r (argumento) | DN en p ∨ r | 23-24 Extraordinaria |
| ⊢ (¬p → q) → ¬q → p | DN en p | 23-24 Móstoles |
| ⊢ ¬(p → q) → p ∧ ¬q | LEM en p | 22-23 Móstoles |
| ⊢ (¬p → p) → p | LEM en p | 22-23 Vicálvaro |
| ⊢ p ∧ ¬p → q sin usar ⊥E | DN en q | 22-23 Vicálvaro Especial |
| ⊢ (¬q → ¬p) → (p → q) | LEM en q y DN en q | 21-22 Vicálvaro |

### Ejercicios nuevos (solución abajo; inténtalos antes de mirar)

1. ⊢ p ∧ q → q ∧ p
2. ⊢ (p → q) → ¬q → ¬p
3. ⊢ ¬(p ∨ q) → ¬p
4. ⊢ (p → ¬p) → ¬p
5. ⊢ p ∧ (q ∨ r) → (p ∧ q) ∨ (p ∧ r)
6. ⊢ ¬¬p → p, usando LEM en p

<details>
<summary>Soluciones</summary>

```scala
// 1
def proof1[P, Q]: ((P, Q)) => (Q, P) =
    (p, q) => (q, p)

// 2
def proof2[P, Q]: (P => Q) => Not[Q] => Not[P] =
    f => nq => p => nq(f(p))

// 3
def proof3[P, Q]: Not[Either[P, Q]] => Not[P] =
    npq => p => npq(Left(p))

// 4
def proof4[P]: (P => Not[P]) => Not[P] =
    f => p => f(p)(p)

// 5
def proof5[P, Q, R]: ((P, Either[Q, R])) => Either[(P, Q), (P, R)] =
    (p, e) => e match
        case Left(q)  => Left((p, q))
        case Right(r) => Right((p, r))

// 6
def proof6[P](lem: ExcludedMiddle): Not[Not[P]] => P =
    nnp => lem[P] match
        case Left(p)   => p
        case Right(np) => nnp(np)    // Nothing → vale como P
```

</details>

Siguiente: **03-isomorfismos-tipos-algebraicos.ipynb**.